# GC002：消费账本、算子计量与冻结候选
复用已发布且哈希核验的 GPU KV/推理检查，不重复已完成的工程测量。仅恢复原始数据资产并重建期望消费流；历史实际逐 Update 输入哈希未记录，不能事后证明。
新增 4 模型 × Batch 4 × TBPTT 64 的合成算子清单检查：零学习率，核验模型权重不变，不消费真实训练 Token。明确区分标量操作参考口径和硬件 FLOPs；未知浮点算子阻断 Compute-Matched。
启用原有 Internet、KAGGLE_API_TOKEN、GITHUB_TOKEN；有 GPU 时验证该环境的运算覆盖，无 GPU 不做 CPU 吞吐 Benchmark。Save & Run All 自动发布。冻结候选不是正式冻结或训练授权。


In [ ]:
from pathlib import Path
import json, os, signal, subprocess, sys, time

BRANCH = 'codex/epclm-gc002-fair-architecture'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-epclm-core-001'
HARD_LIMIT_SECONDS = 7200
deadline = time.monotonic() + HARD_LIMIT_SECONDS

if not os.environ.get('KAGGLE_API_TOKEN', '').strip():
    from kaggle_secrets import UserSecretsClient
    os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN').strip()

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix':
            os.killpg(process.pid, signal.SIGTERM)
        else:
            process.terminate()
        try:
            output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix':
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Hard two-hour limit reached. Kaggle Output remains under ' + '/kaggle/working/epclm-gc002-preparation')
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Kaggle Output remains under ' + '/kaggle/working/epclm-gc002-preparation')

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'reset', '--hard', 'FETCH_HEAD'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH, REPOSITORY, str(REPO)])

run_bounded([sys.executable, str(REPO / "research/projects/cellular-language-models/experiments/epclm-core-001/gc002-fair-architecture/run.py"), "--prepare-only", "--output", "/kaggle/working/epclm-gc002-preparation", "--device", ("cuda" if __import__("torch").cuda.is_available() else "cpu"), "--recover-input-assets", "--publish"], cwd=REPO, timeout_seconds=6000)
